# ELEC5305 Final Experiment - VS Code


## 1. Project folder


In [ ]:
from pathlib import Path
import sys

assert (3, 9) <= sys.version_info[:2] <= (3, 12), "Select a Python 3.12 kernel."
PROJECT_ROOT = Path(r"D:\download\5305\ELEC5305_Final")
assert (PROJECT_ROOT / "data/splits/development_manifest.csv").is_file()
assert (PROJECT_ROOT / "data/splits/test_manifest.csv").is_file()
assert (PROJECT_ROOT / "results/final/development/mixture_manifest.csv").is_file()
assert (PROJECT_ROOT / "results/final/test/mixture_manifest.csv").is_file()

print("Project root:", PROJECT_ROOT)


In [ ]:
%pip install -q torch==2.9.1 --index-url https://download.pytorch.org/whl/cu128
%pip install -q openai-whisper==20250625 jiwer==4.0.0 scipy pandas soundfile matplotlib pystoi imageio-ffmpeg


In [ ]:
import os
import shutil
import imageio_ffmpeg

CODE_ROOT = PROJECT_ROOT / "vscode_code"
CODE_ROOT.mkdir(parents=True, exist_ok=True)
tools_dir = PROJECT_ROOT / "tools"
tools_dir.mkdir(exist_ok=True)
ffmpeg_path = tools_dir / "ffmpeg.exe"
if not ffmpeg_path.is_file():
    shutil.copy2(imageio_ffmpeg.get_ffmpeg_exe(), ffmpeg_path)
os.environ["PATH"] = str(tools_dir) + os.pathsep + os.environ["PATH"]
print("Code folder:", CODE_ROOT)


## Project Python code


### `run_deepfilternet.py`


In [ ]:
%%writefile D:/download/5305/ELEC5305_Final/vscode_code/run_deepfilternet.py
"""Run DeepFilterNet3 on the saved noisy audio."""

import argparse
import hashlib
import json
import shutil
import subprocess
import time
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import soundfile as sf
from scipy.signal import resample_poly


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("project_root", type=Path)
    parser.add_argument("split", choices=["development", "test"])
    parser.add_argument("--binary", type=Path, help="Optional DeepFilterNet v0.5.6 executable path")
    args = parser.parse_args()

    root = args.project_root.resolve()
    default_name = (
        "deep-filter-0.5.6-x86_64-pc-windows-msvc.exe"
        if platform.system() == "Windows"
        else "deep-filter-0.5.6-x86_64-unknown-linux-musl"
    )
    binary = args.binary.resolve() if args.binary else root / "tools" / default_name
    if not binary.is_file():
        raise FileNotFoundError(binary)

    result_root = root / "results" / "final" / args.split
    manifest = pd.read_csv(result_root / "mixture_manifest.csv")
    manifest = manifest.rename(
        columns={
            "utteranceId": "utterance_id",
            "noiseType": "noise_type",
            "targetSNRdB": "target_snr_db",
            "noisyRelativePath": "noisy_relative_path",
        }
    )
    processing_root = result_root / "deepfilter_processing"
    processing_root.mkdir(parents=True, exist_ok=True)
    old_path = result_root / "deepfilter_manifest.csv"
    old = pd.read_csv(old_path) if old_path.is_file() else pd.DataFrame()
    old_by_key = {
        (row.utterance_id, row.noise_type, int(row.target_snr_db)): row
        for row in old.itertuples()
    }

    rows: list[dict[str, object]] = []
    for index, item in enumerate(manifest.itertuples(), 1):
        noisy_path = root / Path(item.noisy_relative_path)
        output_path = noisy_path.with_name(noisy_path.name.replace("_noisy.wav", "_deepfilter.wav"))
        key = (item.utterance_id, item.noise_type, int(item.target_snr_db))

        noisy, sample_rate = sf.read(noisy_path, dtype="float32")
        if sample_rate != 16000 or noisy.ndim != 1:
            raise ValueError(f"Expected mono 16 kHz audio: {noisy_path}")

        elapsed = float("nan")
        if not output_path.is_file():
            work_dir = processing_root / item.utterance_id / f"{item.noise_type}_{int(item.target_snr_db)}dB"
            input_dir = work_dir / "input"
            enhanced_dir = work_dir / "enhanced"
            input_dir.mkdir(parents=True, exist_ok=True)
            enhanced_dir.mkdir(parents=True, exist_ok=True)
            input_path = input_dir / "input_48k.wav"

            audio_48k = resample_poly(noisy, 3, 1).astype(np.float32, copy=False)
            original_length_48k = len(audio_48k)
            padded = np.concatenate([audio_48k, np.zeros(48000, dtype=np.float32)])
            sf.write(input_path, padded, 48000, subtype="FLOAT")

            started = time.perf_counter()
            process = subprocess.run(
                [str(binary), "-D", "-o", str(enhanced_dir), str(input_path)],
                capture_output=True,
                text=True,
            )
            elapsed = time.perf_counter() - started
            if process.returncode != 0:
                raise RuntimeError(f"DeepFilterNet failed for {key}:\n{process.stderr}")

            enhanced_path = enhanced_dir / input_path.name
            enhanced_48k, enhanced_rate = sf.read(enhanced_path, dtype="float32")
            if enhanced_rate != 48000 or enhanced_48k.ndim != 1:
                raise ValueError(f"Unexpected DeepFilterNet output: {enhanced_path}")
            enhanced_48k = enhanced_48k[:original_length_48k]
            enhanced_16k = resample_poly(enhanced_48k, 1, 3).astype(np.float32, copy=False)
            if len(enhanced_16k) != len(noisy):
                raise ValueError(f"DeepFilterNet length mismatch: {key}")
            if not np.isfinite(enhanced_16k).all():
                raise ValueError(f"DeepFilterNet produced invalid samples: {key}")
            sf.write(output_path, enhanced_16k, 16000, subtype="FLOAT")
            shutil.rmtree(work_dir)
        elif key in old_by_key:
            elapsed = float(old_by_key[key].processing_seconds)

        enhanced, enhanced_rate = sf.read(output_path, dtype="float32")
        if enhanced_rate != 16000 or len(enhanced) != len(noisy):
            raise ValueError(f"Saved DeepFilterNet output failed validation: {output_path}")
        rows.append(
            {
                "utterance_id": item.utterance_id,
                "project_split": args.split,
                "noise_type": item.noise_type,
                "target_snr_db": int(item.target_snr_db),
                "input_samples": len(noisy),
                "output_samples": len(enhanced),
                "output_peak": float(np.max(np.abs(enhanced))),
                "processing_seconds": elapsed,
                "real_time_factor": elapsed / (len(noisy) / 16000),
                "deepfilter_relative_path": output_path.relative_to(root).as_posix(),
            }
        )
        print(f"[{index}/{len(manifest)}] {key} complete", flush=True)

    pd.DataFrame(rows).to_csv(old_path, index=False)
    settings = {
        "release": "v0.5.6",
        "model": "DeepFilterNet3",
        "model_source": "embedded in official v0.5.6 binary",
        "binary": binary.name,
        "binary_sha256": sha256(binary),
        "resampling": "scipy.signal.resample_poly",
        "scipy_version": scipy.__version__,
        "sample_rates": "16000 -> 48000 -> 16000",
        "delay_compensation": True,
        "tail_padding_seconds": 1,
        "post_filter": False,
        "output_format": "32-bit float WAV",
    }
    (result_root / "deepfilter_settings.json").write_text(
        json.dumps(settings, indent=2), encoding="utf-8"
    )


if __name__ == "__main__":
    main()


### `run_whisper_asr.py`


In [ ]:
%%writefile D:/download/5305/ELEC5305_Final/vscode_code/run_whisper_asr.py
"""Run Whisper base.en and save progress after each file."""

import argparse
import csv
import json
import math
import time
from importlib.metadata import version
from pathlib import Path

import jiwer
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import whisper
from whisper.normalizers import EnglishTextNormalizer


BASE_METHOD_COLUMNS = {
    "noisy": "noisyRelativePath",
    "spectral_subtraction": "spectralRelativePath",
    "wiener": "wienerRelativePath",
}
ALPHAS = [0.75, 1.00, 1.25, 1.50, 2.00]


def alpha_label(alpha: float) -> str:
    return f"spectral_alpha_{alpha:.2f}".replace(".", "p")


def alpha_filename(stem: str, alpha: float) -> str:
    token = f"{alpha:.2f}".replace(".", "p")
    return f"{stem}_spectral_alpha_{token}.wav"


def load_audio(path: Path) -> np.ndarray:
    audio, sample_rate = sf.read(path, dtype="float32")
    if sample_rate != 16000 or audio.ndim != 1 or not np.isfinite(audio).all():
        raise ValueError(f"Invalid Whisper input: {path}")
    return audio


def resume_snr(value: object) -> int | None:
    """Use one value for the missing SNR in clean-speech rows."""
    return None if pd.isna(value) else int(value)


def build_tasks(root: Path, split: str, include_alpha_sweep: bool, modified_alpha: float | None):
    split_manifest = pd.read_csv(root / "data" / "splits" / f"{split}_manifest.csv")
    mixture = pd.read_csv(root / "results" / "final" / split / "mixture_manifest.csv")
    deepfilter = pd.read_csv(root / "results" / "final" / split / "deepfilter_manifest.csv")
    deepfilter_map = {
        (row.utterance_id, row.noise_type, int(row.target_snr_db)): row.deepfilter_relative_path
        for row in deepfilter.itertuples()
    }
    reference_map = dict(zip(split_manifest.utterance_id, split_manifest.reference_text))
    tasks = []
    for row in split_manifest.itertuples():
        tasks.append(
            {
                "utterance_id": row.utterance_id,
                "noise_type": "clean",
                "target_snr_db": math.nan,
                "method": "clean",
                "path": root / row.clean_wav,
                "reference_text": row.reference_text,
            }
        )

    for row in mixture.itertuples():
        key = (row.utteranceId, row.noiseType, int(row.targetSNRdB))
        for method, column in BASE_METHOD_COLUMNS.items():
            tasks.append(
                {
                    "utterance_id": row.utteranceId,
                    "noise_type": row.noiseType,
                    "target_snr_db": int(row.targetSNRdB),
                    "method": method,
                    "path": root / getattr(row, column),
                    "reference_text": reference_map[row.utteranceId],
                }
            )
        tasks.append(
            {
                "utterance_id": row.utteranceId,
                "noise_type": row.noiseType,
                "target_snr_db": int(row.targetSNRdB),
                "method": "deepfilternet3",
                "path": root / deepfilter_map[key],
                "reference_text": reference_map[row.utteranceId],
            }
        )
        stem = f"{row.noiseType}_{int(row.targetSNRdB)}dB"
        utterance_dir = root / "results" / "final" / split / "audio" / row.utteranceId
        if include_alpha_sweep:
            for alpha in ALPHAS:
                tasks.append(
                    {
                        "utterance_id": row.utteranceId,
                        "noise_type": row.noiseType,
                        "target_snr_db": int(row.targetSNRdB),
                        "method": alpha_label(alpha),
                        "path": utterance_dir / alpha_filename(stem, alpha),
                        "reference_text": reference_map[row.utteranceId],
                    }
                )
        if modified_alpha is not None:
            tasks.append(
                {
                    "utterance_id": row.utteranceId,
                    "noise_type": row.noiseType,
                    "target_snr_db": int(row.targetSNRdB),
                    "method": "spectral_subtraction_modified",
                    "path": utterance_dir / alpha_filename(stem, modified_alpha),
                    "reference_text": reference_map[row.utteranceId],
                }
            )
    return tasks


def summarize(transcripts: pd.DataFrame, result_root: Path) -> None:
    normalizer = EnglishTextNormalizer()

    def score_group(group: pd.DataFrame) -> dict[str, float | int]:
        refs = [normalizer(value) for value in group.reference_text]
        hyps = [normalizer(value) for value in group.hypothesis_text]
        score = jiwer.process_words(refs, hyps)
        return {
            "wer_percent": score.wer * 100,
            "substitutions": score.substitutions,
            "deletions": score.deletions,
            "insertions": score.insertions,
            "reference_words": score.hits + score.substitutions + score.deletions,
            "mean_asr_seconds": group.asr_seconds.mean(),
            "mean_asr_rtf": group.asr_rtf.mean(),
        }

    condition_rows = []
    for keys, group in transcripts.groupby(["noise_type", "target_snr_db", "method"], dropna=False):
        noise, snr, method = keys
        condition_rows.append({"noise_type": noise, "target_snr_db": snr, "method": method, **score_group(group)})
    by_condition = pd.DataFrame(condition_rows)
    noisy_lookup = {
        (row.noise_type, row.target_snr_db): row.wer_percent
        for row in by_condition.itertuples()
        if row.method == "noisy"
    }
    by_condition["delta_wer_pp"] = [
        (row.wer_percent - noisy_lookup[(row.noise_type, row.target_snr_db)])
        if row.method != "clean" and (row.noise_type, row.target_snr_db) in noisy_lookup
        else math.nan
        for row in by_condition.itertuples()
    ]
    by_condition["relative_wer_change_percent"] = [
        100 * row.delta_wer_pp / noisy_lookup[(row.noise_type, row.target_snr_db)]
        if not math.isnan(row.delta_wer_pp) and noisy_lookup[(row.noise_type, row.target_snr_db)] != 0
        else math.nan
        for row in by_condition.itertuples()
    ]
    by_condition.to_csv(result_root / "wer_summary_by_condition.csv", index=False)

    overall_rows = []
    for method, group in transcripts[transcripts.method != "clean"].groupby("method", sort=False):
        overall_rows.append({"method": method, **score_group(group)})
    overall = pd.DataFrame(overall_rows)
    noisy_overall = float(overall.loc[overall.method == "noisy", "wer_percent"].iloc[0])
    overall["delta_wer_pp"] = overall.wer_percent - noisy_overall
    overall["relative_wer_change_percent"] = 100 * overall.delta_wer_pp / noisy_overall
    overall.to_csv(result_root / "wer_summary_overall.csv", index=False)


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("project_root", type=Path)
    parser.add_argument("split", choices=["development", "test"])
    parser.add_argument("--include-alpha-sweep", action="store_true")
    parser.add_argument("--modified-alpha", type=float)
    args = parser.parse_args()

    root = args.project_root.resolve()
    result_root = root / "results" / "final" / args.split / "asr"
    result_root.mkdir(parents=True, exist_ok=True)
    transcript_path = result_root / "transcripts.csv"
    existing = pd.read_csv(transcript_path) if transcript_path.is_file() else pd.DataFrame()
    done = {
        (row.utterance_id, row.noise_type, resume_snr(row.target_snr_db), row.method)
        for row in existing.itertuples()
    }

    device = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(5305)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(5305)
    model = whisper.load_model("base.en", device=device)
    model.eval()
    decode_settings = {
        "language": "en",
        "task": "transcribe",
        "temperature": 0.0,
        "beam_size": 5,
        "fp16": False,
        "condition_on_previous_text": False,
        "compression_ratio_threshold": 2.4,
        "logprob_threshold": -1.0,
        "no_speech_threshold": 0.6,
        "verbose": None,
    }

    tasks = build_tasks(root, args.split, args.include_alpha_sweep, args.modified_alpha)
    rows = existing.to_dict("records") if not existing.empty else []
    for index, task in enumerate(tasks, 1):
        key = (
            task["utterance_id"],
            task["noise_type"],
            resume_snr(task["target_snr_db"]),
            task["method"],
        )
        if key in done:
            continue
        path = Path(task["path"])
        if not path.is_file():
            raise FileNotFoundError(path)
        audio = load_audio(path)
        started = time.perf_counter()
        with torch.inference_mode():
            result = model.transcribe(audio, **decode_settings)
        elapsed = time.perf_counter() - started
        rows.append(
            {
                "utterance_id": task["utterance_id"],
                "project_split": args.split,
                "noise_type": task["noise_type"],
                "target_snr_db": task["target_snr_db"],
                "method": task["method"],
                "reference_text": task["reference_text"],
                "hypothesis_text": result["text"].strip(),
                "audio_seconds": len(audio) / 16000,
                "asr_seconds": elapsed,
                "asr_rtf": elapsed / (len(audio) / 16000),
                "audio_relative_path": path.relative_to(root).as_posix(),
            }
        )
        pd.DataFrame(rows).to_csv(transcript_path, index=False, encoding="utf-8-sig")
        print(f"[{index}/{len(tasks)}] {key} -> {result['text'].strip()}", flush=True)

    transcripts = pd.DataFrame(rows)
    summarize(transcripts, result_root)
    settings = {
        "model": "base.en",
        "device": device,
        "decode_settings": decode_settings,
        "normalizer": "Whisper EnglishTextNormalizer",
        "openai_whisper_version": version("openai-whisper"),
        "jiwer_version": version("jiwer"),
        "torch_version": torch.__version__,
        "cuda_runtime": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "seed": 5305,
    }
    (result_root / "recognition_settings.json").write_text(
        json.dumps(settings, indent=2), encoding="utf-8"
    )


if __name__ == "__main__":
    main()


### `compute_final_logmel_distance.py`


In [ ]:
%%writefile D:/download/5305/ELEC5305_Final/vscode_code/compute_final_logmel_distance.py
"""Compare Whisper log-Mel features with the clean reference."""

import argparse
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import whisper


def read_audio(path: Path) -> np.ndarray:
    audio, sample_rate = sf.read(path, dtype="float32")
    if sample_rate != 16000 or audio.ndim != 1 or not np.isfinite(audio).all():
        raise ValueError(f"Expected finite mono 16 kHz audio: {path}")
    return audio


def mel(audio: np.ndarray) -> torch.Tensor:
    return whisper.log_mel_spectrogram(whisper.pad_or_trim(audio), n_mels=80)


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("project_root", type=Path)
    parser.add_argument("split", choices=["development", "test"])
    parser.add_argument("--include-modified", action="store_true")
    args = parser.parse_args()

    root = args.project_root.resolve()
    result_root = root / "results" / "final" / args.split
    mixtures = pd.read_csv(result_root / "mixture_manifest.csv")
    deep = pd.read_csv(result_root / "deepfilter_manifest.csv")
    deep_map = {
        (row.utterance_id, row.noise_type, int(row.target_snr_db)): row.deepfilter_relative_path
        for row in deep.itertuples()
    }
    modified_map: dict[tuple[str, str, int], str] = {}
    if args.include_modified:
        modified = pd.read_csv(result_root / "modified_spectral_manifest.csv")
        modified_map = {
            (row.utterance_id, row.noise_type, int(row.target_snr_db)): row.output_relative_path
            for row in modified.itertuples()
        }

    rows: list[dict[str, object]] = []
    for index, row in enumerate(mixtures.itertuples(), 1):
        key = (row.utteranceId, row.noiseType, int(row.targetSNRdB))
        clean = read_audio(root / row.cleanRelativePath) * float(row.commonScale)
        clean_mel = mel(clean)
        valid_frames = min(clean_mel.shape[-1], int(np.ceil(len(clean) / whisper.audio.HOP_LENGTH)))
        reference = clean_mel[:, :valid_frames]
        paths = {
            "noisy": row.noisyRelativePath,
            "spectral_subtraction": row.spectralRelativePath,
            "wiener": row.wienerRelativePath,
            "deepfilternet3": deep_map[key],
        }
        if args.include_modified:
            paths["spectral_subtraction_modified"] = modified_map[key]
        for method, relative_path in paths.items():
            processed = read_audio(root / relative_path)
            if len(processed) != len(clean):
                raise ValueError(f"Length mismatch: {key} {method}")
            difference = mel(processed)[:, :valid_frames] - reference
            rows.append(
                {
                    "utterance_id": row.utteranceId,
                    "noise_type": row.noiseType,
                    "target_snr_db": int(row.targetSNRdB),
                    "method": method,
                    "valid_logmel_frames": valid_frames,
                    "logmel_mse": float(torch.mean(difference.square()).item()),
                    "logmel_mae": float(torch.mean(torch.abs(difference)).item()),
                    "audio_relative_path": relative_path,
                }
            )
        if index % 25 == 0 or index == len(mixtures):
            print(f"[{index}/{len(mixtures)}] mixtures", flush=True)

    per_file = pd.DataFrame(rows)
    summary = (
        per_file.groupby(["noise_type", "target_snr_db", "method"], sort=False)
        .agg(
            n_files=("utterance_id", "count"),
            mean_logmel_mse=("logmel_mse", "mean"),
            std_logmel_mse=("logmel_mse", "std"),
            mean_logmel_mae=("logmel_mae", "mean"),
        )
        .reset_index()
    )
    per_file.to_csv(result_root / "logmel_per_file.csv", index=False)
    summary.to_csv(result_root / "logmel_summary.csv", index=False)
    print(summary.to_string(index=False))


if __name__ == "__main__":
    main()


### `select_spectral_alpha.py`


In [ ]:
%%writefile D:/download/5305/ELEC5305_Final/vscode_code/select_spectral_alpha.py
"""Choose spectral subtraction alpha using only the development set."""

import argparse
import json
import re
from pathlib import Path

import pandas as pd


def parse_alpha(method: str) -> float:
    match = re.fullmatch(r"spectral_alpha_(\d+)p(\d+)", method)
    if not match:
        raise ValueError(method)
    return float(f"{match.group(1)}.{match.group(2)}")


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("project_root", type=Path)
    args = parser.parse_args()
    root = args.project_root.resolve()
    dev_root = root / "results" / "final" / "development"
    wer = pd.read_csv(dev_root / "asr" / "wer_summary_overall.csv")
    metrics = pd.read_csv(dev_root / "metrics_per_file.csv")

    candidates = wer[wer.method.str.startswith("spectral_alpha_")].copy()
    candidates["alpha"] = candidates.method.map(parse_alpha)
    mean_metrics = (
        metrics[metrics.method.str.startswith("spectral_alpha_")]
        .groupby("method", as_index=False)
        .agg(
            mean_stoi=("stoiScore", "mean"),
            mean_output_snr_db=("outputSNRdB", "mean"),
        )
    )
    candidates = candidates.merge(mean_metrics, on="method", how="left")
    candidates["distance_from_original_alpha"] = (candidates.alpha - 1.0).abs()
    candidates = candidates.sort_values(
        ["wer_percent", "mean_stoi", "distance_from_original_alpha", "alpha"],
        ascending=[True, False, True, True],
    )
    selected = candidates.iloc[0]
    output = {
        "selected_alpha": float(selected.alpha),
        "selection_split": "development",
        "primary_rule": "lowest corpus-level WER across all development noise/SNR conditions",
        "tie_break_1": "highest mean STOI",
        "tie_break_2": "smallest absolute distance from original alpha=1.0",
        "candidate_table": candidates.to_dict("records"),
    }
    (dev_root / "selected_spectral_alpha.json").write_text(
        json.dumps(output, indent=2), encoding="utf-8"
    )
    candidates.to_csv(dev_root / "spectral_alpha_selection.csv", index=False)
    print(candidates.to_string(index=False))
    print(f"Selected alpha: {selected.alpha:.2f}")


if __name__ == "__main__":
    main()


### `analyze_and_plot.py`


In [ ]:
%%writefile D:/download/5305/ELEC5305_Final/vscode_code/analyze_and_plot.py
"""Make the result tables, plots, and example cases."""

import argparse
import json
import shutil
from pathlib import Path

import jiwer
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
from scipy import signal
from whisper.normalizers import EnglishTextNormalizer


METHODS = [
    "noisy",
    "spectral_subtraction",
    "wiener",
    "deepfilternet3",
    "spectral_subtraction_modified",
]
LABELS = {
    "clean": "Clean",
    "noisy": "No enhancement",
    "spectral_subtraction": "Spectral subtraction (α=1.0)",
    "wiener": "Wiener",
    "deepfilternet3": "DeepFilterNet3",
    "spectral_subtraction_modified": "Spectral subtraction (α=1.5)",
}
COLORS = {
    "noisy": "#4D4D4D",
    "spectral_subtraction": "#0072B2",
    "wiener": "#009E73",
    "deepfilternet3": "#D55E00",
    "spectral_subtraction_modified": "#CC79A7",
}


def save_figure(fig: plt.Figure, output: Path) -> None:
    fig.savefig(output.with_suffix(".png"), dpi=240, bbox_inches="tight")
    fig.savefig(output.with_suffix(".pdf"), bbox_inches="tight")
    plt.close(fig)


def pair_errors(reference: str, hypothesis: str, normalizer: EnglishTextNormalizer) -> dict[str, float | int]:
    result = jiwer.process_words(normalizer(reference), normalizer(hypothesis))
    words = result.hits + result.substitutions + result.deletions
    return {
        "errors": result.substitutions + result.deletions + result.insertions,
        "reference_words": words,
        "utterance_wer_percent": 100 * result.wer,
        "substitutions": result.substitutions,
        "deletions": result.deletions,
        "insertions": result.insertions,
    }


def make_case_spectrogram(case: pd.Series, transcripts: pd.DataFrame, root: Path, output: Path) -> None:
    keys = ["utterance_id", "noise_type", "target_snr_db"]
    subset = transcripts.copy()
    for key in keys:
        subset = subset[subset[key] == case[key]]
    noisy = subset[subset.method == "noisy"].iloc[0]
    enhanced = subset[subset.method == case.method].iloc[0]
    mixture = pd.read_csv(root / "results" / "final" / "test" / "mixture_manifest.csv")
    match = mixture[
        (mixture.utteranceId == case.utterance_id)
        & (mixture.noiseType == case.noise_type)
        & (mixture.targetSNRdB == case.target_snr_db)
    ].iloc[0]
    paths = [
        ("Clean", root / match.cleanRelativePath),
        ("No enhancement", root / noisy.audio_relative_path),
        (LABELS[case.method], root / enhanced.audio_relative_path),
    ]
    fig, axes = plt.subplots(3, 1, figsize=(9, 7), sharex=True, sharey=True)
    for axis, (title, path) in zip(axes, paths):
        audio, fs = sf.read(path, dtype="float32")
        frequencies, times, spectrum = signal.spectrogram(audio, fs, nperseg=512, noverlap=384)
        use = frequencies <= 8000
        axis.pcolormesh(times, frequencies[use] / 1000, 10 * np.log10(spectrum[use] + 1e-10), shading="auto", cmap="magma", vmin=-100, vmax=-25)
        axis.set_ylabel("kHz")
        axis.set_title(title, loc="left", fontsize=10)
    axes[-1].set_xlabel("Time (s)")
    fig.suptitle(f"{case.case_type.title()} case: {case.utterance_id}, {case.noise_type}, {int(case.target_snr_db)} dB")
    fig.tight_layout()
    save_figure(fig, output)


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("project_root", type=Path)
    args = parser.parse_args()
    root = args.project_root.resolve()
    result = root / "results" / "final" / "test"
    figures = result / "figures"
    examples = result / "examples"
    figures.mkdir(parents=True, exist_ok=True)
    examples.mkdir(parents=True, exist_ok=True)

    transcripts = pd.read_csv(result / "asr" / "transcripts.csv")
    if len(transcripts) != 2300 or transcripts.duplicated(["utterance_id", "noise_type", "target_snr_db", "method"]).any():
        raise ValueError("Expected exactly 2,300 unique completed test recognition rows.")
    condition = pd.read_csv(result / "asr" / "wer_summary_by_condition.csv")
    overall = pd.read_csv(result / "asr" / "wer_summary_overall.csv")
    metrics = pd.read_csv(result / "metrics_summary.csv")
    metrics_per_file = pd.read_csv(result / "metrics_per_file.csv")
    logmel = pd.read_csv(result / "logmel_summary.csv")
    condition_metrics = condition.merge(metrics, on=["noise_type", "target_snr_db", "method"], how="left").merge(
        logmel, on=["noise_type", "target_snr_db", "method"], how="left"
    )
    condition_metrics.to_csv(result / "combined_condition_summary.csv", index=False)

    clean_wer = float(condition.loc[condition.method == "clean", "wer_percent"].iloc[0])
    fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.1), sharey=True)
    for axis, noise in zip(axes, ["white", "office", "traffic"]):
        subset = condition[condition.noise_type == noise]
        for method in METHODS:
            rows = subset[subset.method == method].sort_values("target_snr_db")
            axis.plot(rows.target_snr_db, rows.wer_percent, marker="o", linewidth=2, color=COLORS[method], label=LABELS[method])
        axis.axhline(clean_wer, color="#777777", linestyle="--", linewidth=1.3, label=f"Clean ({clean_wer:.1f}%)")
        axis.set_title(noise.title())
        axis.set_xlabel("Input SNR (dB)")
        axis.set_xticks([0, 5, 10])
        axis.grid(alpha=0.25)
    axes[0].set_ylabel("Corpus WER (%)")
    handles, labels = axes[-1].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.16), frameon=False)
    fig.suptitle("Whisper base.en WER across noise types and input SNRs")
    fig.tight_layout()
    fig.subplots_adjust(bottom=0.25)
    save_figure(fig, figures / "01_wer_vs_snr")

    enhanced = [m for m in METHODS if m != "noisy"]
    pivot = condition[condition.method.isin(enhanced)].pivot(index=["noise_type", "target_snr_db"], columns="method", values="delta_wer_pp")
    pivot = pivot.reindex(pd.MultiIndex.from_product([["white", "office", "traffic"], [0, 5, 10]], names=pivot.index.names))
    fig, axes = plt.subplots(1, 4, figsize=(13.5, 4.4), sharey=True)
    for axis, method in zip(axes, enhanced):
        image = pivot[method].unstack().loc[["white", "office", "traffic"], [0, 5, 10]]
        mesh = axis.imshow(image, cmap="RdBu_r", vmin=-8, vmax=8, aspect="auto")
        axis.set_title(LABELS[method], fontsize=9)
        axis.set_xticks(range(3), ["0", "5", "10"])
        axis.set_yticks(range(3), ["White", "Office", "Traffic"])
        axis.set_xlabel("Input SNR (dB)")
        for y in range(3):
            for x in range(3):
                axis.text(x, y, f"{image.iloc[y, x]:+.1f}", ha="center", va="center", fontsize=9)
    color_axis = fig.add_axes([0.925, 0.20, 0.015, 0.58])
    fig.colorbar(mesh, cax=color_axis, label="ΔWER (percentage points)")
    fig.suptitle("Change from matched unenhanced noisy speech (negative is better)")
    fig.subplots_adjust(left=0.07, right=0.90, bottom=0.15, top=0.84, wspace=0.28)
    save_figure(fig, figures / "02_delta_wer_heatmap")

    fig, axes = plt.subplots(2, 3, figsize=(13.5, 7.2), sharex=True)
    for column, noise in enumerate(["white", "office", "traffic"]):
        subset = metrics[metrics.noise_type == noise]
        for method in METHODS:
            rows = subset[subset.method == method].sort_values("target_snr_db")
            axes[0, column].plot(rows.target_snr_db, rows.mean_stoi, marker="o", color=COLORS[method], label=LABELS[method])
            axes[1, column].plot(rows.target_snr_db, rows.mean_output_snr_db, marker="o", color=COLORS[method])
        axes[0, column].set_title(noise.title())
        axes[0, column].grid(alpha=0.25)
        axes[1, column].grid(alpha=0.25)
        axes[1, column].set_xlabel("Input SNR (dB)")
        axes[1, column].set_xticks([0, 5, 10])
    axes[0, 0].set_ylabel("Mean STOI")
    axes[1, 0].set_ylabel("Mean output SNR (dB)")
    handles, labels = axes[0, -1].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.06), frameon=False)
    fig.suptitle("Signal metrics by method and condition")
    fig.tight_layout()
    fig.subplots_adjust(bottom=0.13)
    save_figure(fig, figures / "03_stoi_and_output_snr")

    alpha = condition[condition.method.isin(["spectral_subtraction", "spectral_subtraction_modified"])]
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), sharey=True)
    for axis, noise in zip(axes, ["white", "office", "traffic"]):
        subset = alpha[alpha.noise_type == noise]
        for method in ["spectral_subtraction", "spectral_subtraction_modified"]:
            rows = subset[subset.method == method].sort_values("target_snr_db")
            axis.plot(rows.target_snr_db, rows.wer_percent, marker="o", linewidth=2, color=COLORS[method], label=LABELS[method])
        axis.set_title(noise.title())
        axis.set_xlabel("Input SNR (dB)")
        axis.set_xticks([0, 5, 10])
        axis.grid(alpha=0.25)
    axes[0].set_ylabel("Corpus WER (%)")
    handles, labels = axes[-1].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.1), frameon=False)
    fig.suptitle("Test-set evaluation of the development-selected spectral-subtraction modification")
    fig.tight_layout()
    fig.subplots_adjust(bottom=0.22)
    save_figure(fig, figures / "04_spectral_modification")

    scatter = condition_metrics[condition_metrics.method.isin(METHODS)].copy()
    fig, axis = plt.subplots(figsize=(7.2, 5.2))
    for method in METHODS:
        rows = scatter[scatter.method == method]
        axis.scatter(rows.mean_logmel_mse, rows.wer_percent, s=55, alpha=0.85, color=COLORS[method], label=LABELS[method])
    correlation = scatter[["mean_logmel_mse", "wer_percent"]].corr().iloc[0, 1]
    axis.set_xlabel("Mean official Whisper log-Mel MSE to scaled clean")
    axis.set_ylabel("Corpus WER (%)")
    axis.set_title(f"Feature distance and recognition error by condition (Pearson r={correlation:.2f})")
    axis.grid(alpha=0.25)
    axis.legend(fontsize=8)
    save_figure(fig, figures / "05_logmel_vs_wer")

    timing = (
        metrics_per_file[metrics_per_file.method.isin(METHODS)]
        .groupby("method")
        .realTimeFactor.agg(median_rtf="median", p95_rtf=lambda values: values.quantile(0.95), max_rtf="max")
        .reindex(METHODS)
    )
    timing.to_csv(result / "enhancement_timing_summary.csv")
    asr_rtf = transcripts.groupby("method").asr_rtf.mean()
    fig, axes = plt.subplots(1, 2, figsize=(11, 5.2))
    axes[0].bar([LABELS[m] for m in METHODS], timing.median_rtf, color=[COLORS[m] for m in METHODS])
    axes[0].set_ylabel("Median enhancement RTF")
    axes[0].set_title("Front-end processing\n(median; raw outliers retained in CSV)", fontsize=10)
    axes[1].bar([LABELS[m] for m in METHODS], [asr_rtf[m] for m in METHODS], color=[COLORS[m] for m in METHODS])
    axes[1].set_ylabel("Whisper RTF")
    axes[1].set_title("Recognition")
    for axis in axes:
        axis.tick_params(axis="x", labelrotation=58)
        axis.grid(axis="y", alpha=0.25)
    fig.suptitle("Real-time factors on the experiment machine", y=0.96)
    fig.subplots_adjust(left=0.09, right=0.98, top=0.78, bottom=0.40, wspace=0.35)
    save_figure(fig, figures / "06_runtime_rtf")

    normalizer = EnglishTextNormalizer()
    scored = []
    for row in transcripts[transcripts.method != "clean"].itertuples():
        scored.append({**row._asdict(), **pair_errors(row.reference_text, row.hypothesis_text, normalizer)})
    scored_df = pd.DataFrame(scored)
    utterance_totals = (
        scored_df.groupby(["utterance_id", "method"])[["errors", "reference_words"]]
        .sum()
        .reset_index()
    )
    total_error = utterance_totals.pivot(index="utterance_id", columns="method", values="errors")
    total_words = utterance_totals[utterance_totals.method == "noisy"].set_index("utterance_id").reference_words
    rng = np.random.default_rng(5305)
    ids = total_error.index.to_numpy()
    bootstrap_rows = []
    for method in enhanced:
        estimates = np.empty(10000)
        for replicate in range(len(estimates)):
            sampled = rng.choice(ids, size=len(ids), replace=True)
            numerator = (total_error.loc[sampled, method].to_numpy() - total_error.loc[sampled, "noisy"].to_numpy()).sum()
            denominator = total_words.loc[sampled].to_numpy().sum()
            estimates[replicate] = 100 * numerator / denominator
        observed = 100 * (total_error[method].sum() - total_error["noisy"].sum()) / total_words.sum()
        bootstrap_rows.append(
            {
                "method": method,
                "observed_delta_wer_pp": observed,
                "cluster_bootstrap_ci95_low_pp": np.quantile(estimates, 0.025),
                "cluster_bootstrap_ci95_high_pp": np.quantile(estimates, 0.975),
                "bootstrap_replicates": len(estimates),
                "cluster_unit": "utterance",
            }
        )
    pd.DataFrame(bootstrap_rows).to_csv(result / "delta_wer_cluster_bootstrap.csv", index=False)
    noisy = scored_df[scored_df.method == "noisy"][
        ["utterance_id", "noise_type", "target_snr_db", "errors", "utterance_wer_percent", "hypothesis_text", "audio_relative_path"]
    ].rename(columns={c: f"noisy_{c}" for c in ["errors", "utterance_wer_percent", "hypothesis_text", "audio_relative_path"]})
    candidates = scored_df[scored_df.method.isin(enhanced)].merge(noisy, on=["utterance_id", "noise_type", "target_snr_db"])
    candidates["delta_errors"] = candidates.errors - candidates.noisy_errors
    candidates["delta_utterance_wer_pp"] = candidates.utterance_wer_percent - candidates.noisy_utterance_wer_percent
    success = candidates.sort_values(["delta_errors", "delta_utterance_wer_pp"]).iloc[0].copy()
    failure = candidates.sort_values(["delta_errors", "delta_utterance_wer_pp"], ascending=False).iloc[0].copy()
    cases = []
    for case_type, row in [("success", success), ("failure", failure)]:
        row["case_type"] = case_type
        cases.append(row)
        method_token = row.method.replace("spectral_subtraction_modified", "spectral_modified")
        noisy_source = root / row.noisy_audio_relative_path
        enhanced_source = root / row.audio_relative_path
        shutil.copy2(noisy_source, examples / f"{case_type}_{row.utterance_id}_noisy.wav")
        shutil.copy2(enhanced_source, examples / f"{case_type}_{row.utterance_id}_{method_token}.wav")
    case_table = pd.DataFrame(cases)
    keep = [
        "case_type", "utterance_id", "noise_type", "target_snr_db", "method", "reference_text",
        "noisy_hypothesis_text", "hypothesis_text", "noisy_errors", "errors", "delta_errors",
        "substitutions", "deletions", "insertions",
        "noisy_utterance_wer_percent", "utterance_wer_percent", "delta_utterance_wer_pp",
        "noisy_audio_relative_path", "audio_relative_path",
    ]
    case_table[keep].to_csv(examples / "selected_cases.csv", index=False, encoding="utf-8-sig")

    # Check whether the worst failure alone explains the method-level result.
    compared_methods = ["noisy", failure.method]
    comparison = scored_df[scored_df.method.isin(compared_methods)].copy()
    matched_failure = (
        (comparison.utterance_id == failure.utterance_id)
        & (comparison.noise_type == failure.noise_type)
        & (comparison.target_snr_db == failure.target_snr_db)
    )
    sensitivity_rows = []
    for label, frame in [("all_rows", comparison), ("matched_failure_removed", comparison[~matched_failure])]:
        for method, group in frame.groupby("method"):
            errors = int(group.errors.sum())
            reference_words = int(group.reference_words.sum())
            sensitivity_rows.append(
                {
                    "analysis": label,
                    "method": method,
                    "errors": errors,
                    "reference_words": reference_words,
                    "wer_percent": 100 * errors / reference_words,
                }
            )
    sensitivity = pd.DataFrame(sensitivity_rows)
    noisy_wer = sensitivity[sensitivity.method == "noisy"].set_index("analysis").wer_percent
    sensitivity["delta_wer_pp"] = [
        row.wer_percent - noisy_wer[row.analysis] for row in sensitivity.itertuples()
    ]
    sensitivity.to_csv(examples / "failure_case_sensitivity.csv", index=False)

    for row in case_table.itertuples():
        make_case_spectrogram(pd.Series(row._asdict()), transcripts, root, figures / f"07_{row.case_type}_spectrogram")

    findings = {
        "test_transcription_rows": len(transcripts),
        "clean_wer_percent": clean_wer,
        "overall_methods": overall.to_dict("records"),
        "feature_wer_condition_correlation": float(correlation),
        "selected_success": case_table.iloc[0][keep[:-2]].to_dict(),
        "selected_failure": case_table.iloc[1][keep[:-2]].to_dict(),
        "failure_case_sensitivity": sensitivity.to_dict("records"),
    }
    with (result / "analysis_findings.json").open("w", encoding="utf-8") as handle:
        json.dump(findings, handle, indent=2, ensure_ascii=False, default=str)
    print(overall.to_string(index=False))
    print(case_table[keep[:-2]].to_string(index=False))


if __name__ == "__main__":
    main()


### `validate_final_results.py`


In [ ]:
%%writefile D:/download/5305/ELEC5305_Final/vscode_code/validate_final_results.py
"""Check the saved development and test results."""

import argparse
import json
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("project_root", type=Path)
    args = parser.parse_args()
    root = args.project_root.resolve()

    development = pd.read_csv(root / "data" / "splits" / "development_manifest.csv")
    test = pd.read_csv(root / "data" / "splits" / "test_manifest.csv")
    assert len(development) == 5 and len(test) == 50
    assert set(development.speaker_id).isdisjoint(set(test.speaker_id))

    report: dict[str, object] = {
        "development_utterances": len(development),
        "test_utterances": len(test),
        "development_speakers": development.speaker_id.nunique(),
        "test_speakers": test.speaker_id.nunique(),
        "speaker_overlap": 0,
    }
    for split, expected_mixtures, expected_transcripts, expected_metrics in [
        ("development", 45, 410, 405),
        ("test", 450, 2300, 2250),
    ]:
        result = root / "results" / "final" / split
        mixtures = pd.read_csv(result / "mixture_manifest.csv")
        deep = pd.read_csv(result / "deepfilter_manifest.csv")
        transcripts = pd.read_csv(result / "asr" / "transcripts.csv")
        metrics = pd.read_csv(result / "metrics_per_file.csv")
        assert len(mixtures) == expected_mixtures and len(deep) == expected_mixtures
        assert len(transcripts) == expected_transcripts and len(metrics) == expected_metrics
        assert not transcripts.duplicated(["utterance_id", "noise_type", "target_snr_db", "method"]).any()
        assert not metrics.duplicated(["utteranceId", "noiseType", "targetSNRdB", "method"]).any()
        assert np.max(np.abs(mixtures.measuredInputSNRdB - mixtures.targetSNRdB)) < 0.01
        assert mixtures.reconstructionError.max() < 1e-9
        assert metrics.estimatedLagSamples.abs().max() == 0
        assert metrics.peakAmplitude.max() < 1
        for relative in transcripts.audio_relative_path:
            path = root / relative
            assert path.is_file(), path
            info = sf.info(path)
            assert info.samplerate == 16000 and info.channels == 1
        report[split] = {
            "mixtures": len(mixtures),
            "transcripts": len(transcripts),
            "waveform_metric_rows": len(metrics),
            "max_abs_target_snr_error_db": float(np.max(np.abs(mixtures.measuredInputSNRdB - mixtures.targetSNRdB))),
            "max_reconstruction_error": float(mixtures.reconstructionError.max()),
            "max_abs_alignment_lag_samples": int(metrics.estimatedLagSamples.abs().max()),
            "max_peak_amplitude": float(metrics.peakAmplitude.max()),
        }
    output = root / "results" / "final" / "validation_report.json"
    output.write_text(json.dumps(report, indent=2), encoding="utf-8")
    print(json.dumps(report, indent=2))


if __name__ == "__main__":
    main()


### `compute_waveform_metrics.py`


In [ ]:
%%writefile D:/download/5305/ELEC5305_Final/vscode_code/compute_waveform_metrics.py
"""Compute waveform metrics for the final experiment."""

import argparse
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
from pystoi import stoi


def read_mono(path: Path) -> tuple[np.ndarray, int]:
    audio, sample_rate = sf.read(path, dtype="float64")
    if audio.ndim != 1 or sample_rate != 16000:
        raise ValueError(f"Expected mono 16 kHz audio: {path}")
    if not np.isfinite(audio).all():
        raise ValueError(f"Invalid audio samples: {path}")
    return audio, sample_rate


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("project_root", type=Path)
    parser.add_argument("split", choices=["development", "test"])
    parser.add_argument("--include-alpha-sweep", action="store_true")
    parser.add_argument("--include-modified", action="store_true")
    args = parser.parse_args()

    root = args.project_root.resolve()
    result = root / "results" / "final" / args.split
    mixtures = pd.read_csv(result / "mixture_manifest.csv")
    deep = pd.read_csv(result / "deepfilter_manifest.csv")
    modified = None
    if args.include_modified:
        modified = pd.read_csv(result / "modified_spectral_manifest.csv")

    methods = ["noisy", "spectral_subtraction", "wiener", "deepfilternet3"]
    if args.include_alpha_sweep:
        methods += [
            "spectral_alpha_0p75",
            "spectral_alpha_1p00",
            "spectral_alpha_1p25",
            "spectral_alpha_1p50",
            "spectral_alpha_2p00",
        ]
    if args.include_modified:
        methods.append("spectral_subtraction_modified")

    rows = []
    for index, mixture in enumerate(mixtures.itertuples(), start=1):
        clean, sample_rate = read_mono(root / mixture.cleanRelativePath)
        reference = clean * float(mixture.commonScale)
        duration = len(reference) / sample_rate

        deep_row = deep[
            (deep.utterance_id == mixture.utteranceId)
            & (deep.noise_type == mixture.noiseType)
            & (deep.target_snr_db == mixture.targetSNRdB)
        ].iloc[0]

        for method in methods:
            if method == "noisy":
                relative = mixture.noisyRelativePath
                elapsed = 0.0
            elif method == "spectral_subtraction":
                relative = mixture.spectralRelativePath
                elapsed = float(mixture.spectralSeconds)
            elif method == "wiener":
                relative = mixture.wienerRelativePath
                elapsed = float(mixture.wienerSeconds)
            elif method == "deepfilternet3":
                relative = deep_row.deepfilter_relative_path
                elapsed = float(deep_row.processing_seconds)
            elif method == "spectral_subtraction_modified":
                match = modified[
                    (modified.utterance_id == mixture.utteranceId)
                    & (modified.noise_type == mixture.noiseType)
                    & (modified.target_snr_db == mixture.targetSNRdB)
                ].iloc[0]
                relative = match.output_relative_path
                elapsed = float(match.processing_seconds)
            else:
                token = method.removeprefix("spectral_alpha_")
                stem = f"{mixture.noiseType}_{int(mixture.targetSNRdB)}dB"
                relative = (
                    f"results/final/{args.split}/audio/{mixture.utteranceId}/"
                    f"{stem}_spectral_alpha_{token}.wav"
                )
                elapsed = np.nan

            processed, processed_rate = read_mono(root / relative)
            if processed_rate != sample_rate or len(processed) != len(reference):
                raise ValueError(f"Audio length mismatch: {relative}")

            error_power = np.sum((reference - processed) ** 2)
            output_snr = 10 * np.log10(
                np.sum(reference ** 2) / (error_power + np.finfo(float).eps)
            )
            score = stoi(reference, processed, sample_rate, extended=False)

            rows.append({
                "utteranceId": mixture.utteranceId,
                "noiseType": mixture.noiseType,
                "targetSNRdB": mixture.targetSNRdB,
                "method": method,
                "outputSNRdB": output_snr,
                "snrImprovementdB": output_snr - mixture.measuredInputSNRdB,
                "stoiScore": score,
                "estimatedLagSamples": 0,
                "durationSeconds": duration,
                "processingSeconds": elapsed,
                "realTimeFactor": elapsed / duration if np.isfinite(elapsed) else np.nan,
                "peakAmplitude": float(np.max(np.abs(processed))),
                "audioRelativePath": relative,
            })

        print(f"[{index}/{len(mixtures)}] {mixture.utteranceId}", flush=True)

    per_file = pd.DataFrame(rows)
    per_file.to_csv(result / "metrics_per_file.csv", index=False)

    summary = (
        per_file.groupby(["noiseType", "targetSNRdB", "method"], as_index=False)
        .agg(
            n_files=("outputSNRdB", "size"),
            mean_output_snr_db=("outputSNRdB", "mean"),
            mean_snr_improvement_db=("snrImprovementdB", "mean"),
            mean_stoi=("stoiScore", "mean"),
            max_abs_lag_samples=("estimatedLagSamples", lambda x: int(np.max(np.abs(x)))),
            mean_real_time_factor=("realTimeFactor", "mean"),
        )
        .rename(columns={"noiseType": "noise_type", "targetSNRdB": "target_snr_db"})
    )
    summary.to_csv(result / "metrics_summary.csv", index=False)
    print(summary)


if __name__ == "__main__":
    main()


## DeepFilterNet3


In [ ]:

import os
import subprocess
import sys
import urllib.request
from pathlib import Path

PROJECT_ROOT = Path(r"D:\download\5305\ELEC5305_Final")
assert PROJECT_ROOT.is_dir()

tools_dir = PROJECT_ROOT / "tools"
tools_dir.mkdir(exist_ok=True)
binary = tools_dir / "deep-filter-0.5.6-x86_64-pc-windows-msvc.exe"
binary_url = (
    "https://github.com/Rikorose/DeepFilterNet/releases/download/v0.5.6/"
    "deep-filter-0.5.6-x86_64-pc-windows-msvc.exe"
)
if not binary.exists():
    urllib.request.urlretrieve(binary_url, binary)
subprocess.run([str(binary), "--version"], check=True)


In [ ]:

python = sys.executable
subprocess.run([python, str(CODE_ROOT / "run_deepfilternet.py"),
                str(PROJECT_ROOT), "development", "--binary", str(binary)], check=True)
subprocess.run([python, str(CODE_ROOT / "run_deepfilternet.py"),
                str(PROJECT_ROOT), "test", "--binary", str(binary)], check=True)


## Development-set evaluation and alpha selection


In [ ]:

subprocess.run([python, str(CODE_ROOT / "compute_waveform_metrics.py"),
                str(PROJECT_ROOT), "development", "--include-alpha-sweep"], check=True)
subprocess.run([python, str(CODE_ROOT / "run_whisper_asr.py"),
                str(PROJECT_ROOT), "development", "--include-alpha-sweep"], check=True)
subprocess.run([python, str(CODE_ROOT / "select_spectral_alpha.py"),
                str(PROJECT_ROOT)], check=True)

import json
selection_path = PROJECT_ROOT / "results/final/development/selected_spectral_alpha.json"
selected_alpha = json.loads(selection_path.read_text())["selected_alpha"]
print("Selected alpha:", selected_alpha)
assert abs(selected_alpha - 1.5) < 1e-12, (
    "The development selection changed. Generate the modified test audio in MATLAB again."
)


## Independent test-set evaluation


In [ ]:

subprocess.run([python, str(CODE_ROOT / "compute_waveform_metrics.py"),
                str(PROJECT_ROOT), "test", "--include-modified"], check=True)
subprocess.run([python, str(CODE_ROOT / "run_whisper_asr.py"),
                str(PROJECT_ROOT), "test", "--modified-alpha", "1.5"], check=True)
subprocess.run([python, str(CODE_ROOT / "compute_final_logmel_distance.py"),
                str(PROJECT_ROOT), "test", "--include-modified"], check=True)
subprocess.run([python, str(CODE_ROOT / "analyze_and_plot.py"),
                str(PROJECT_ROOT)], check=True)
subprocess.run([python, str(CODE_ROOT / "validate_final_results.py"),
                str(PROJECT_ROOT)], check=True)


## Final results


In [ ]:

import pandas as pd
from IPython.display import Image, display

test_root = PROJECT_ROOT / "results/final/test"
display(pd.read_csv(test_root / "asr/wer_summary_overall.csv"))
display(pd.read_csv(test_root / "asr/wer_summary_by_condition.csv"))

for figure in sorted((test_root / "figures").glob("*.png")):
    print(figure.name)
    display(Image(filename=str(figure), width=900))


In [ ]:

import zipfile

output_zip = PROJECT_ROOT / "ELEC5305_Final_Results.zip"
with zipfile.ZipFile(output_zip, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in (PROJECT_ROOT / "results/final").rglob("*"):
        if not path.is_file():
            continue
        relative = path.relative_to(PROJECT_ROOT)
        if "audio" in relative.parts or "processing" in relative.parts:
            continue
        archive.write(path, relative.as_posix())

print("Final result ZIP:", output_zip)
